In [29]:
# Imports 
import numpy as np
import pandas as pd
import torch
import os
import grelu.resources
import h5py
import anndata as ad

from grelu.lightning import LightningModel as BaseLightningModel

In [30]:
import grelu.io.genome
genome = grelu.io.genome.get_genome("dm6")
seq_len = 2000     # Length of the input sequence
training = True


In [31]:
# Load the signal matrix from the .h5 file.
h5_path = "combined_all_signals_after_filtering.h5"
with h5py.File(h5_path, "r") as f:
    signal_matrix = f["signal_matrix"][:]

# Load corresponding BED file with region coordinates
bed_path = "filtered_peaks.bed"
bed = pd.read_csv(bed_path, sep="\t", header=None, names=["chrom","start","end"])

trusted_chroms = ['chr2L', 'chr2R', 'chr3L', 'chr3R']
keep_mask = bed['chrom'].isin(trusted_chroms)
bed = bed[keep_mask].reset_index(drop=True)
signal_matrix = signal_matrix[keep_mask.values, :]

assert signal_matrix.shape[0] == len(bed)

# This block will remove any peaks that are too close to the chromosome ends
# to support the requested seq_len of 2000.
print("Pre-filtering peaks based on seq_len to prevent out-of-boundary errors...")

# 1. Get the official chromosome sizes for the dm6 genome 
genome_file = "data/dm6.fa.fai"
chromsizes = {}
with open(genome_file, "r") as f:
    for line in f:
        chrom, size, *_ = line.strip().split("\t")
        chromsizes[chrom] = int(size)
        
expected_len = seq_len


# 1. Define the required margin on each side of a peak's center
margin = seq_len // 2 # For seq_len=2000, this is 1000

# 2. Map the size of each chromosome to every peak in the BED file.
# This creates a new 'chrom_size' column for easy comparison.
bed['chrom_size'] = bed['chrom'].map(chromsizes)

# 3. Calculate the center for all peaks at once (this is fast and vectorized).
bed['center'] = bed['start'] + (bed['end'] - bed['start']) // 2

# 4. Create the final boolean mask with a single, clean operation.
# A peak is kept if its center is within the valid margins of its chromosome.
keep_mask_filtered = (bed['center'] >= margin) & (bed['center'] <= (bed['chrom_size'] - margin))

# 5. Apply the final mask to both your BED dataframe and your signal matrix
original_count = len(bed)
bed = bed[keep_mask_filtered].reset_index(drop=True)
signal_matrix = signal_matrix[keep_mask_filtered.values, :]

print(f"Removed {original_count - len(bed)} peaks that were too close to chromosome ends for seq_len={seq_len}.")
print(f"Kept {len(bed)} valid peaks.")

# Your assertion should still pass
assert signal_matrix.shape[0] == len(bed)



# Construct the AnnData object from the data

# gReLU's data loaders expect an AnnData object where:
# - ad.X contains the signal matrix (transposed to tasks x regions)
# - ad.var contains the region coordinates
# - ad.obs contains the names of the tasks/experiments

# Define the full list of task names corresponding to the original signal_matrix
all_task_names = [
    "orer_gaf_mbt_nexus", "zldrnai_15to2_atac", "orer_dl_mbt_nexus",
    "zldrnai_1to15_atac", "orer_zld_mbt_nexus", "gd7_1to15_atac",
    "orer_h3k27ac_mbt_seq", "gd7_15to2_atac", "orer_2to25_atac",
    "orer_cad_mbt_nexus", "zldrnai_25to3_atac", "gd7_2to25_atac",
    "orer_wce_mbt_seq", "orer_1to15_atac", "orer_twi_mbt_nexus",
    "cic6_2to25_atac", "orer_25to3_atac", "zldrnai_2to25_atac",
    "orer_bcd_mbt_nexus", "cic6_25to3_atac", "orer_15to2_atac",
    "gd7_25to3_atac", "orer_h3k4me1_mbt_seq", "orer_mbt_100u_mnase"
]

# Create a pandas Series to easily find indices
all_tasks_series = pd.Series(all_task_names)

# Find the names and original indices of ATAC-seq tasks
is_atac_mask = all_tasks_series.str.contains("atac")
atac_task_names = all_tasks_series[is_atac_mask].tolist()
atac_indices = all_tasks_series.index[is_atac_mask].tolist()

print(f"Found and selected {len(atac_task_names)} ATAC-seq tasks.")

# Filter the original signal_matrix to keep only the ATAC-seq columns
signal_matrix_atac = signal_matrix[:, atac_indices]


# Construct the AnnData object from the filtered ATAC-seq data
print("Creating AnnData object...")

# The signal matrix needs to be in the shape (n_tasks, n_regions)
X = signal_matrix_atac.T # Use the filtered matrix

# The .var attribute holds the information about the genomic regions
var = bed

# The .obs attribute holds information about the tasks (experiments).
obs = pd.DataFrame({"task_name" : atac_task_names}) # Use the filtered names

#Create the AnnData object
ad = ad.AnnData(X, obs=obs, var=var)


#Set the index for .var and .obs to be compliant with gReLU
ad.var.index = ad.var["chrom"] + "_" + ad.var["start"].astype(str) + "_" + ad.var["end"].astype(str)
ad.obs.index = ad.obs["task_name"]

print("AnnData object created with shape (tasks, regions):", ad.shape)
print("Variable (regoin) info example:\n", ad.var.head())
print("\nObservations (task) info example: \n", ad.obs.head())


# Download the genome and split data
genome = "dm6"

# Split chr2R into validation and test sets
# Step 1: Randomly split chr2R into val/test
chr2R_mask = ad.var["chrom"] == "chr2R"
chr2R_indices = np.where(chr2R_mask)[0]
np.random.shuffle(chr2R_indices)
val_indices = chr2R_indices[:len(chr2R_indices)//2]
test_indices = chr2R_indices[len(chr2R_indices)//2:]

# Step 2: Temporarily rename chromosome for splitting
ad.var.iloc[val_indices, ad.var.columns.get_loc("chrom")] = "chr2R_val"
ad.var.iloc[test_indices, ad.var.columns.get_loc("chrom")] = "chr2R_test"

# Split the AnnData object using gReLU's utility function
ad_train, ad_val, ad_test = grelu.data.preprocess.split(
    ad,
    train_chroms=['chr2L', 'chr3L', 'chr3R'],
    val_chroms=['chr2R_val'],
    test_chroms=['chr2R_test']
)

# Restore real chromosome names
for d in [ad_train, ad_val, ad_test]:
    d.var["chrom"] = d.var["chrom"].replace({
        "chr2R_val": "chr2R",
        "chr2R_test": "chr2R"
    })


# Create labeled sequence datasets
# This gReLU class will automatically handle fetching DNA sequences for the
# coordinates in ad.var and pairing them with the target signals in ad.X.

print("\nCreating PyTorch datasets...")

import grelu.data.dataset

# Data augmentation is applied to improve model robustness

train_dataset = grelu.data.dataset.AnnDataSeqDataset(
    ad_train.copy(),
    genome=genome,
    seq_len=seq_len,
    rc=True, # Augmentation: randomly reverse-complement the sequence
    max_seq_shift=0, # Augmentation: randomly shift the sequence by up to max_seq_shift bp, however, intervals were already padded in my data.
)

# No augmentation for validation and test sets
val_dataset = grelu.data.dataset.AnnDataSeqDataset(ad_val.copy(), genome=genome, seq_len=seq_len)
test_dataset = grelu.data.dataset.AnnDataSeqDataset(ad_test.copy(), genome=genome, seq_len=seq_len)


Pre-filtering peaks based on seq_len to prevent out-of-boundary errors...
Removed 13 peaks that were too close to chromosome ends for seq_len=2000.
Kept 368502 valid peaks.
Found and selected 14 ATAC-seq tasks.
Creating AnnData object...


/home/fgsasse_lrs_2/miniconda3/envs/lab-rotation/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/fgsasse_lrs_2/miniconda3/envs/lab-rotation/lib/python3.9/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


AnnData object created with shape (tasks, regions): (14, 368502)
Variable (regoin) info example:
                  chrom  start   end  chrom_size  center
chr2L_926_1126   chr2L    926  1126    23513712    1026
chr2L_1126_1326  chr2L   1126  1326    23513712    1226
chr2L_1326_1526  chr2L   1326  1526    23513712    1426
chr2L_1526_1726  chr2L   1526  1726    23513712    1626
chr2L_1726_1926  chr2L   1726  1926    23513712    1826

Observations (task) info example: 
                              task_name
task_name                             
zldrnai_15to2_atac  zldrnai_15to2_atac
zldrnai_1to15_atac  zldrnai_1to15_atac
gd7_1to15_atac          gd7_1to15_atac
gd7_15to2_atac          gd7_15to2_atac
orer_2to25_atac        orer_2to25_atac
Selecting training samples
Keeping 281434 intervals


Selecting validation samples
Keeping 43534 intervals


Selecting test samples
Keeping 43534 intervals
Final sizes: train: (14, 281434), val: (14, 43534), test: (14, 43534)

Creating PyTorch datasets...


/tmp/ipykernel_2960569/3100217092.py:142: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  d.var["chrom"] = d.var["chrom"].replace({


In [ ]:
# Use the pretrained enformer model (Borzoi model with published weights)
# Note: the genome is dm6, not human or mouse. This may yield suboptimal results.

model_params = {
    "model_type":"EnformerPretrainedModel",
    "n_tasks":ad.shape[0],       # Change to the task count, 14 atac tasks
    "n_transformers":11,         
    "crop_len":0, # How many base pairs you trim from each end of the input sequence before computing the loss. Since the head outputs L=1 (we avg-pool before the MLP), use: (seq_len - 1)//2
    "final_pool_func":None       # How to pool the final sequence representations to get fixed-size task predictions. Options: "avg", "max", None


}

""" The number of transformer blocks stacked in the Enformer “transformer tower
    Each block = multi-head self-attention + feed-forward sublayers.
	More blocks → larger receptive field for long-range interactions, but heavier GPU cost. """

train_params = {
    "task": "regression",  # binary, multiclass, or regression
    "loss": "poisson",
    "lr": 1e-4,
    "optimizer": "adam",
    "batch_size": 256,
    "num_workers": 8,
    "devices": [0],
    "logger": "csv",
    "save_dir": "enformer_results/enformer",
    "max_epochs": 30,
    "checkpoint": True,
    "clip": 0,
    "pos_weight": None,
    "class_weights": None,
    "total_weight": None,
    "accumulate_grad_batches": 1,

}



In [33]:
class LightningModelXY(LightningModel):
    def _xy(self, batch):
        if isinstance(batch, (list, tuple)):
            if len(batch) >= 2:
                return batch[0], batch[1]
            raise ValueError(f"Batch has <2 elements: {len(batch)}")
        if isinstance(batch, dict):
            x = batch.get("x", batch.get("inputs"))
            y = batch.get("y", batch.get("labels", batch.get("target")))
            if x is None or y is None:
                raise ValueError(f"Dict batch missing x/y keys: {batch.keys()}")
            return x, y
        raise ValueError(f"Unsupported batch type: {type(batch)}")

    def training_step(self, batch, batch_idx):
        x, y = self._xy(batch)
        logits = self.forward(x, logits=True)
        loss = self.loss(logits, y)
        self.log("train_loss", loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = self._xy(batch)
        logits = self.forward(x, logits=True)
        loss = self.loss(logits, y)
        y_hat = self.activation(logits)
        self.log("val_loss", loss, on_epoch=True)
        self.update_metrics(self.val_metrics, y_hat, y)
        self.val_losses.append(loss)
        return loss

    def test_step(self, batch, batch_idx):
        x, y = self._xy(batch)
        logits = self.forward(x, logits=True)
        loss = self.loss(logits, y)
        y_hat = self.activation(logits)
        self.log("test_loss", loss, on_epoch=True)
        self.update_metrics(self.test_metrics, y_hat, y)
        self.test_losses.append(loss)
        return loss

In [34]:
from grelu.model.heads import MLPHead
from grelu.lightning import LightningModel
import torch.nn as nn

lit = LightningModelXY(model_params, train_params)


# build an MLP head and a wrapper that pools length -> 1
class AvgPoolThenMLP(nn.Module):
    def __init__(self, mlp_head: nn.Module):
        super().__init__()
        self.mlp = mlp_head
    def forward(self, x):           # x: (B, C, L)
        x = x.mean(dim=-1, keepdim=True)   # (B, C, 1)
        return self.mlp(x)          # MLPHead expects (B, C, 1)

mlp_head = MLPHead(
    n_tasks=ad.shape[0],
    in_channels=3072,   # Enformer/Borzoi trunk channels
    in_len=1,           # because we avg-pool to length=1
    hidden_size=[512, 256],
    act_func = "gelu",
    norm=True,
    dropout=0.1,
    dtype=None,            
    device=None
)

lit.model.head = AvgPoolThenMLP(mlp_head)



wandb: Downloading large artifact human_state_dict:latest, 939.29MB. 1 files... 
wandb:   1 of 1 files downloaded.  
Done. 0:0:1.2


In [35]:
print("model seq_len:", getattr(lit.model.embedding, "seq_len", "unknown"))

model seq_len: unknown


In [36]:
from torch.utils.data import DataLoader
x, y = next(iter(DataLoader(val_dataset, batch_size=4)))
with torch.no_grad():
    logits = lit.forward(x, logits=True)
print("logits shape:", logits.shape)  # (B, 14, 1)

logits shape: torch.Size([4, 14, 1])


In [40]:
preds_val = lit.predict_on_dataset(val_dataset, devices=[0], batch_size=256)
print(preds_val.shape)  # (N_val, 14, 1)

for d in (train_dataset, val_dataset, test_dataset):
    if hasattr(d, "predict"):
        d.predict = False

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Predicting DataLoader 0: 100%|██████████| 171/171 [01:08<00:00,  2.48it/s]
(43534, 14, 1)


In [41]:
# Optional: name tasks so rows are labeled
lit.data_params["tasks"] = {"name": list(ad.obs.index)}  # your 14 ATAC names

val_metrics  = lit.test_on_dataset(val_dataset,  devices=[0], batch_size=256)
print(val_metrics.head())

test_metrics = lit.test_on_dataset(test_dataset, devices=[0], batch_size=256)
print(test_metrics.head())

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Testing DataLoader 0: 100%|██████████| 171/171 [01:09<00:00,  2.47it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss            136.7404022216797
        test_mse             3145.36181640625
      test_pearson         -0.04502077028155327
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


                       test_mse  test_pearson
zldrnai_15to2_atac  2379.853271     -0.072129
zldrnai_1to15_atac   246.954559      0.105156
gd7_1to15_atac      1038.683716     -0.062296
gd7_15to2_atac      2629.024170     -0.036198
orer_2to25_atac     4140.561035     -0.002197


/home/fgsasse_lrs_2/miniconda3/envs/lab-rotation/lib/python3.9/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:425: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=63` in the `DataLoader` to improve performance.


Testing DataLoader 0: 100%|██████████| 171/171 [01:09<00:00,  2.47it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_loss           136.19065856933594
        test_mse             3113.427490234375
      test_pearson         -0.04725712910294533
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                       test_mse  test_pearson
zldrnai_15to2_atac  2384.319824     -0.076898
zldrnai_1to15_atac   246.536591      0.104919
gd7_1to15_atac      1029.269409     -0.055979
gd7_15to2_atac      2605.101318     -0.034310
orer_2to25_atac     4085.714600     -0.007437


In [ ]:
# (Optional) head-only fine-tune, then unfreeze
# # freeze trunk
for p in lit.model.embedding.parameters():
    p.requires_grad = False

lit.train_params["lr"] = 1e-3  # head can take a bigger LR
trainer = lit.train_on_dataset(train_dataset, val_dataset)

# unfreeze
for p in lit.model.embedding.parameters():
    p.requires_grad = True
 
lit.train_params["lr"] = 1e-4  # smaller LR for trunk
trainer = lit.train_on_dataset(train_dataset, val_dataset)

# re-evaluate
val_after = lit.test_on_dataset(val_dataset, devices=[0], batch_size=32)
print(val_after.head())

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Validation DataLoader 0: 100%|██████████| 171/171 [01:07<00:00,  2.54it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        val_loss             136.7404022216797
         val_mse             3145.36181640625
       val_pearson         -0.04502077028155327
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type                    | Params | Mode
----------------------------------------------------------------
0 | model        | EnformerPretrainedModel | 231 M  | eval
1 | loss         | PoissonNLLLoss          | 0      | eval
2 | val_metrics  | MetricCollection        | 0      | eval
3 | test_metrics | MetricCollection        | 0      | eval
4 | transform    | Identity                | 0      | eval
----------------------------------------------------------------
1.7 M     Trainable params
229 M     Non-trainable params
231 M     Total params
926.246   Total estimated model params size (MB)
0         Modules in train mode
553       Modules in eval mode


Epoch 29:  18%|█▊        | 399/2199 [02:42<12:11,  2.46it/s, v_num=0, train_loss_step=3.400, train_loss_epoch=3.470] 